In [1]:
import os
import gzip
from pathlib import Path

raw_dir = Path("../data/raw")

# Look at what's in the activities folder
activity_files = list((raw_dir / "activities").glob("*"))
print(f"Total activity files: {len(activity_files)}")

# Show the first 10 and their extensions
for f in sorted(activity_files)[:10]:
    print(f.name, f"  ({f.stat().st_size // 1024} KB)")

Total activity files: 1402
10000952539.fit   (14 KB)
10000952539.fit:Zone.Identifier   (0 KB)
10046118394.fit   (27 KB)
10046118394.fit:Zone.Identifier   (0 KB)
10078375825.gpx   (391 KB)
10078375825.gpx:Zone.Identifier   (0 KB)
10107949856.gpx   (184 KB)
10107949856.gpx:Zone.Identifier   (0 KB)
10113345452.gpx   (368 KB)
10113345452.gpx:Zone.Identifier   (0 KB)


In [2]:
import fitparse
import gzip

def open_fit(filepath):
    """Open a .fit or .fit.gz file and return a FitFile object."""
    filepath = Path(filepath)
    if filepath.suffix == '.gz':
        with gzip.open(filepath, 'rb') as f:
            data = f.read()
        return fitparse.FitFile(data)  # pass bytes directly
    else:
        return fitparse.FitFile(str(filepath))

# Pick the first .fit.gz file
sample_file = sorted((raw_dir / "activities").glob("*.fit.gz"))[0]
print("Parsing:", sample_file.name)

fit = open_fit(sample_file)

# What message types exist in this file?
message_types = {}
for msg in fit.get_messages():
    t = msg.name
    message_types[t] = message_types.get(t, 0) + 1

print("\nMessage types found:")
for name, count in sorted(message_types.items(), key=lambda x: -x[1]):
    print(f"  {name}: {count} messages")

Parsing: 11147594123.fit.gz

Message types found:
  unknown_233: 569 messages
  gps_metadata: 569 messages
  record: 154 messages
  unknown_325: 33 messages
  device_info: 21 messages
  unknown_326: 11 messages
  unknown_216: 6 messages
  lap: 5 messages
  event: 4 messages
  workout_step: 3 messages
  unknown_113: 3 messages
  file_id: 1 messages
  file_creator: 1 messages
  unknown_288: 1 messages
  unknown_327: 1 messages
  unknown_22: 1 messages
  unknown_141: 1 messages
  device_settings: 1 messages
  user_profile: 1 messages
  unknown_79: 1 messages
  sport: 1 messages
  unknown_13: 1 messages
  zones_target: 1 messages
  training_file: 1 messages
  workout: 1 messages
  exercise_title: 1 messages
  unknown_140: 1 messages
  session: 1 messages
  activity: 1 messages


In [3]:
# Reset and look at the fields inside one record message
fit = open_fit(sample_file)
records = list(fit.get_messages("record"))

print(f"Total record messages: {len(records)}")
print(f"\nFields in first record:")
for field in records[0]:
    print(f"  {field.name}: {field.value}  (units: {field.units})")

Total record messages: 154

Fields in first record:
  cadence: 51  (units: rpm)
  distance: 0.0  (units: m)
  enhanced_altitude: 672.0  (units: m)
  enhanced_speed: 0.961  (units: m/s)
  fractional_cadence: 0.0  (units: rpm)
  heart_rate: 87  (units: bpm)
  position_lat: 638583254  (units: semicircles)
  position_long: -1353965603  (units: semicircles)
  timestamp: 2023-12-21 14:54:30  (units: None)
  unknown_135: 176  (units: None)
  unknown_136: 87  (units: None)
  unknown_87: 0  (units: None)


In [4]:
import fitparse
import gzip
import pandas as pd
import numpy as np
from pathlib import Path

# Semicircles → degrees conversion constant
SEMICIRCLE_TO_DEG = 180 / (2 ** 31)

# The fields we want from each record message
# We specify these explicitly — don't just grab everything,
# because unknown_* fields vary between files and will cause
# schema mismatches when you stack 700 DataFrames later
RECORD_FIELDS = {
    'timestamp',
    'heart_rate',
    'cadence',
    'fractional_cadence',
    'enhanced_speed',
    'enhanced_altitude',
    'distance',
    'position_lat',
    'position_long',
}

def parse_fit_file(filepath):
    """
    Parse a single .fit or .fit.gz file.
    Returns a DataFrame of per-second record messages,
    or None if the file has no usable records.
    """
    filepath = Path(filepath)

    # Handle both compressed and uncompressed .fit files
    try:
        if filepath.suffix == '.gz':
            with gzip.open(filepath, 'rb') as f:
                raw = f.read()
            fit = fitparse.FitFile(raw)
        else:
            fit = fitparse.FitFile(str(filepath))
    except Exception as e:
        print(f"  [SKIP] Could not open {filepath.name}: {e}")
        return None

    rows = []
    for msg in fit.get_messages("record"):
        row = {}
        for field in msg:
            if field.name in RECORD_FIELDS and field.value is not None:
                row[field.name] = field.value
        # Only keep rows that have at least a timestamp and speed
        # (some records are partial — watch buffering artifacts)
        if 'timestamp' in row and 'enhanced_speed' in row:
            rows.append(row)

    if len(rows) < 10:
        # Too few records to be useful (probably a sensor calibration activity)
        return None

    df = pd.DataFrame(rows)

    # Convert semicircles to decimal degrees
    # We use .get() pattern because not all records have GPS (indoor runs)
    if 'position_lat' in df.columns:
        df['lat'] = df['position_lat'] * SEMICIRCLE_TO_DEG
        df['lon'] = df['position_long'] * SEMICIRCLE_TO_DEG
        df.drop(columns=['position_lat', 'position_long'], inplace=True)

    # Combine cadence + fractional_cadence for full precision
    # e.g. cadence=85, fractional=0.5 → true cadence = 85.5 steps/min
    if 'cadence' in df.columns and 'fractional_cadence' in df.columns:
        df['cadence'] = df['cadence'] + df['fractional_cadence'].fillna(0)
        df.drop(columns=['fractional_cadence'], inplace=True)

    # Convert speed (m/s) to pace (min/km) — more intuitive for running
    # Guard against zero speed (standing still) which would give inf pace
    df['pace_min_km'] = np.where(
        df['enhanced_speed'] > 0.1,          # moving threshold: ~0.36 km/h
        1000 / (df['enhanced_speed'] * 60),   # the conversion
        np.nan                                 # standing still → NaN, not inf
    )

    # Add activity ID from filename (we'll use this to join with activities.csv later)
    df['activity_id'] = filepath.stem.replace('.fit', '')

    # Set timestamp as index — makes time-series operations much cleaner later
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    df.sort_index(inplace=True)

    return df

# Test on your sample file
sample_file = sorted((raw_dir / "activities").glob("*.fit.gz"))[0]
df_sample = parse_fit_file(sample_file)

if df_sample is not None:
    print(f"Shape: {df_sample.shape}")
    print(f"Duration: {len(df_sample)} seconds ({len(df_sample)/60:.1f} min)")
    print(f"\nColumns: {list(df_sample.columns)}")
    print(f"\nFirst 3 rows:")
    print(df_sample.head(3))
    print(f"\nNull counts:")
    print(df_sample.isnull().sum())

Shape: (154, 9)
Duration: 154 seconds (2.6 min)

Columns: ['cadence', 'distance', 'enhanced_altitude', 'enhanced_speed', 'heart_rate', 'lat', 'lon', 'pace_min_km', 'activity_id']

First 3 rows:
                     cadence  distance  enhanced_altitude  enhanced_speed  \
timestamp                                                                   
2023-12-21 14:54:30     51.0      0.00              672.0           0.961   
2023-12-21 14:54:31     51.0      0.90              672.0           0.000   
2023-12-21 14:54:33     51.0      3.55              672.0           0.924   

                     heart_rate        lat         lon  pace_min_km  \
timestamp                                                             
2023-12-21 14:54:30          87  53.525430 -113.488086    17.343045   
2023-12-21 14:54:31          86  53.525427 -113.488107          NaN   
2023-12-21 14:54:33          86  53.525439 -113.488143    18.037518   

                     activity_id  
timestamp                    

In [5]:
# Peek at a GPX file's raw structure so we know what we're parsing
sample_gpx = sorted((raw_dir / "activities").glob("*.gpx"))[0]

with open(sample_gpx, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(line, end='')
        if i > 40:
            print("...")
            break

<?xml version="1.0" encoding="UTF-8"?>
<gpx creator="StravaGPX" version="1.1" xmlns="http://www.topografix.com/GPX/1/1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.topografix.com/GPX/1/1 http://www.topografix.com/GPX/1/1/gpx.xsd http://www.garmin.com/xmlschemas/GpxExtensions/v3 http://www.garmin.com/xmlschemas/GpxExtensionsv3.xsd http://www.garmin.com/xmlschemas/TrackPointExtension/v1 http://www.garmin.com/xmlschemas/TrackPointExtensionv1.xsd" xmlns:gpxtpx="http://www.garmin.com/xmlschemas/TrackPointExtension/v1" xmlns:gpxx="http://www.garmin.com/xmlschemas/GpxExtensions/v3">
 <metadata>
  <time>2023-10-21T14:03:23Z</time>
 </metadata>
 <trk>
  <name>Morning Run</name>
  <type>running</type>
  <trkseg>
   <trkpt lat="52.7340350" lon="-107.5470730">
    <ele>595.9</ele>
    <time>2023-10-21T14:03:23Z</time>
    <extensions>
     <gpxtpx:TrackPointExtension>
      <gpxtpx:hr>99</gpxtpx:hr>
     </gpxtpx:TrackPointExtension>
    </extensions>
   </

In [6]:
import pandas as pd

acts = pd.read_csv(raw_dir / "activities.csv")

print(f"Shape: {acts.shape}")
print(f"\nColumns:\n{list(acts.columns)}")
print(f"\nActivity types:")
print(acts['Activity Type'].value_counts().head(15))

Shape: (706, 103)

Columns:
['Activity ID', 'Activity Date', 'Activity Name', 'Activity Type', 'Activity Description', 'Elapsed Time', 'Distance', 'Max Heart Rate', 'Relative Effort', 'Commute', 'Activity Private Note', 'Activity Gear', 'Filename', 'Athlete Weight', 'Bike Weight', 'Elapsed Time.1', 'Moving Time', 'Distance.1', 'Max Speed', 'Average Speed', 'Elevation Gain', 'Elevation Loss', 'Elevation Low', 'Elevation High', 'Max Grade', 'Average Grade', 'Average Positive Grade', 'Average Negative Grade', 'Max Cadence', 'Average Cadence', 'Max Heart Rate.1', 'Average Heart Rate', 'Max Watts', 'Average Watts', 'Calories', 'Max Temperature', 'Average Temperature', 'Relative Effort.1', 'Total Work', 'Number of Runs', 'Uphill Time', 'Downhill Time', 'Other Time', 'Perceived Exertion', 'Type', 'Start Time', 'Weighted Average Power', 'Power Count', 'Prefer Perceived Exertion', 'Perceived Relative Effort', 'Commute.1', 'Total Weight Lifted', 'From Upload', 'Grade Adjusted Distance', 'Weather

In [7]:
# Filter to running activities only
# Strava uses several type names — check yours
RUN_TYPES = {'Run', 'Trail Run', 'Treadmill Run', 'Virtual Run'}
runs = acts[acts['Activity Type'].isin(RUN_TYPES)].copy()
print(f"Total activities: {len(acts)}")
print(f"Running activities: {len(runs)}")

# Look at what columns actually have data
# Many columns in activities.csv are mostly empty
print("\nColumn fill rates (% non-null):")
fill_rates = (runs.notna().sum() / len(runs) * 100).sort_values(ascending=False)
print(fill_rates[fill_rates > 20].to_string())  # only show columns with >20% data


Total activities: 706
Running activities: 211

Column fill rates (% non-null):
Activity ID                    100.000000
Activity Date                  100.000000
Activity Name                  100.000000
Activity Type                  100.000000
Elapsed Time                   100.000000
Distance                       100.000000
Filename                       100.000000
Elevation Low                  100.000000
Elevation Loss                 100.000000
Elevation Gain                 100.000000
Average Speed                  100.000000
Max Speed                      100.000000
Distance.1                     100.000000
Moving Time                    100.000000
Elapsed Time.1                 100.000000
Calories                       100.000000
Average Grade                  100.000000
Max Grade                      100.000000
Elevation High                 100.000000
Dirt Distance                  100.000000
Commute.1                      100.000000
Grade Adjusted Distance        100.0000

In [10]:
# First, see what the raw values look like
print(runs_clean[['distance_m', 'moving_time_s', 'elapsed_time_s']].head(5))
print("\ndtypes:")
print(runs_clean[['distance_m', 'moving_time_s', 'elapsed_time_s']].dtypes)

  distance_m  moving_time_s  elapsed_time_s
0       6.80         2643.0            2992
1       5.87         2232.0            2275
2      10.50         3956.0            4611
3       6.72         3283.0            5062
4       5.02         1616.0            1907

dtypes:
distance_m            str
moving_time_s     float64
elapsed_time_s      int64
dtype: object


In [28]:
sample_gpx = sorted((raw_dir / "activities").glob("*.gpx"))[0]
tree = ET.parse(sample_gpx)
root = tree.getroot()

points = root.findall('.//gpx:trkpt', NS)

for pt in points[:5]:
    lat = float(pt.attrib['lat'])
    lon = float(pt.attrib['lon'])
    ele = pt.find('gpx:ele', NS).text
    t   = pt.find('gpx:time', NS).text
    print(f"  lat={lat:.7f}  lon={lon:.7f}  ele={ele}  t={t}")

  lat=52.7340350  lon=-107.5470730  ele=595.9  t=2023-10-21T14:03:23Z
  lat=52.7340280  lon=-107.5470820  ele=595.9  t=2023-10-21T14:03:24Z
  lat=52.7340160  lon=-107.5470980  ele=595.9  t=2023-10-21T14:03:25Z
  lat=52.7339820  lon=-107.5471260  ele=595.9  t=2023-10-21T14:03:26Z
  lat=52.7339690  lon=-107.5471340  ele=596.0  t=2023-10-21T14:03:27Z


In [29]:
import xml.etree.ElementTree as ET

# Namespace URIs — expanded from the xmlns declarations in the file header
NS = {
    'gpx':    'http://www.topografix.com/GPX/1/1',
    'gpxtpx': 'http://www.garmin.com/xmlschemas/TrackPointExtension/v1',
}

def parse_gpx_file(filepath):
    """
    Parse a Strava GPX file into a per-second DataFrame.
    Returns None if file is unreadable or too short.
    """
    filepath = Path(filepath)
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"  [SKIP] Bad XML in {filepath.name}: {e}")
        return None

    rows = []
    # Walk every trackpoint in every track segment
    for trkpt in root.findall('.//gpx:trkpt', NS):
        row = {}

        # lat/lon are attributes on the element itself, already in decimal degrees
        # (unlike .fit which uses semicircles)
        try:
            row['lat'] = float(trkpt.attrib['lat'])
            row['lon'] = float(trkpt.attrib['lon'])
        except (KeyError, ValueError):
            continue  # skip points with no coordinates

        # Elevation — direct child element
        ele = trkpt.find('gpx:ele', NS)
        if ele is not None and ele.text:
            row['enhanced_altitude'] = float(ele.text)

        # Timestamp — direct child element, UTC ISO format
        time_el = trkpt.find('gpx:time', NS)
        if time_el is not None and time_el.text:
            row['timestamp'] = pd.to_datetime(time_el.text, utc=True)
        else:
            continue  # no timestamp → useless point

        # HR and cadence live inside extensions → TrackPointExtension
        ext = trkpt.find('.//gpxtpx:TrackPointExtension', NS)
        if ext is not None:
            hr_el  = ext.find('gpxtpx:hr',  NS)
            cad_el = ext.find('gpxtpx:cad', NS)
            if hr_el  is not None and hr_el.text:
                row['heart_rate'] = int(hr_el.text)
            if cad_el is not None and cad_el.text:
                row['cadence'] = float(cad_el.text)

        rows.append(row)

    if len(rows) < 10:
        return None

    df = pd.DataFrame(rows)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df.set_index('timestamp', inplace=True)
    df.sort_index(inplace=True)

    # Compute speed and pace from GPS coordinates + time
    # GPX doesn't store speed directly — we derive it from position deltas
    # haversine distance between consecutive points ÷ time delta
    df['enhanced_speed'] = _compute_speed_from_gps(df)
    df['pace_min_km'] = np.where(
        df['enhanced_speed'] > 0.1,
        1000 / (df['enhanced_speed'] * 60),
        np.nan
    )

    df['activity_id'] = filepath.stem
    return df


# ── Fix 2: robust GPX speed calculation ──────────────────────────────────────
def _compute_speed_from_gps(df):
    lat = np.radians(df['lat'].values)
    lon = np.radians(df['lon'].values)

    lat_prev = np.concatenate([[lat[0]], lat[:-1]])
    lon_prev = np.concatenate([[lon[0]], lon[:-1]])

    dlat = lat - lat_prev
    dlon = lon - lon_prev

    a = (np.sin(dlat/2)**2
         + np.cos(lat_prev) * np.cos(lat) * np.sin(dlon/2)**2)
    dist_m = 2 * 6_371_000 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    dist_m[0] = 0

    idx = df.index
    if idx.tz is not None:
        idx = idx.tz_convert('UTC').tz_localize(None)
    times_s = idx.astype(np.int64) // 1_000_000_000
    dt = np.diff(times_s, prepend=times_s[0])
    dt[0] = 1
    dt = np.where(dt <= 0, 1, dt)

    raw = dist_m / dt
    speed = pd.Series(raw, index=df.index).rolling(5, center=True, min_periods=1).median()
    return np.clip(speed.values, 0, 7)


# Retest
df_gpx = parse_gpx_file(sample_gpx)
print(f"\nPace stats:")
print(df_gpx['pace_min_km'].describe().round(2))
if df_gpx is not None:
    print(f"\nPace range: {df_gpx['pace_min_km'].min():.1f} – {df_gpx['pace_min_km'].max():.1f} min/km")
    print(df_gpx['pace_min_km'].describe().round(2))

if df_gpx is not None:
    print(f"Shape: {df_gpx.shape}")
    print(f"Columns: {list(df_gpx.columns)}")
    print(f"\nFirst 3 rows:")
    print(df_gpx.head(3))
    print(f"\nPace range: {df_gpx['pace_min_km'].min():.1f} – {df_gpx['pace_min_km'].max():.1f} min/km")
    print(f"HR nulls: {df_gpx['heart_rate'].isna().sum()} / {len(df_gpx)}")

  dist_m[:5]: [0.    0.986 1.715 4.225 1.543]
  dt[:5]:     [1 1 1 1 1]
  raw m/s[:5]: [0.    0.986 1.715 4.225 1.543]

Pace stats:
count    1570.00
mean        5.94
std         2.56
min         2.38
25%         4.54
50%         5.15
75%         6.21
max        16.90
Name: pace_min_km, dtype: float64

Pace range: 2.4 – 16.9 min/km
count    1570.00
mean        5.94
std         2.56
min         2.38
25%         4.54
50%         5.15
75%         6.21
max        16.90
Name: pace_min_km, dtype: float64
Shape: (1570, 7)
Columns: ['lat', 'lon', 'enhanced_altitude', 'heart_rate', 'enhanced_speed', 'pace_min_km', 'activity_id']

First 3 rows:
                                 lat         lon  enhanced_altitude  \
timestamp                                                             
2023-10-21 14:03:23+00:00  52.734035 -107.547073              595.9   
2023-10-21 14:03:24+00:00  52.734028 -107.547082              595.9   
2023-10-21 14:03:25+00:00  52.734016 -107.547098              595.9   

  

In [30]:
def clean_numeric(series):
    """
    Convert a column that might be string with commas (e.g. '1,234.5')
    or already numeric into a proper float series.
    """
    if series.dtype == object:  # object dtype = pandas for "string or mixed"
        return pd.to_numeric(series.str.replace(',', '', regex=False), errors='coerce')
    return pd.to_numeric(series, errors='coerce')

# Apply to all numeric columns
NUMERIC_COLS = ['distance_m', 'moving_time_s', 'elapsed_time_s',
                'elevation_gain_m', 'hr_mean', 'hr_max', 'cadence_mean',
                'avg_speed_ms', 'calories', 'relative_effort',
                'grade_adj_distance_m']

for col in NUMERIC_COLS:
    if col in runs_clean.columns:
        runs_clean[col] = clean_numeric(runs_clean[col])

# Now retry the derived columns
runs_clean = runs_clean.rename(columns={'distance_m': 'distance_km_raw'})
runs_clean['distance_km'] = clean_numeric(runs_clean['distance_km_raw'])
runs_clean['moving_time_min'] = runs_clean['moving_time_s'] / 60

runs_clean['avg_pace_min_km'] = np.where(
    runs_clean['avg_speed_ms'] > 0,
    1000 / (runs_clean['avg_speed_ms'] * 60),
    np.nan
)

print(runs_clean[['date','name','distance_km','moving_time_min','avg_pace_min_km','hr_mean']].head(10))

# Silence the date parsing warning with an explicit format attempt
# Strava uses mixed formats across exports so we let pandas infer but suppress noise
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    runs_clean['date'] = pd.to_datetime(runs_clean['date'])

print(f"\nDate range: {runs_clean['date'].min().date()} → {runs_clean['date'].max().date()}")
print(f"Null HR: {runs_clean['hr_mean'].isna().sum()} / {len(runs_clean)} runs")
print(f"\ndtypes after fix:")
print(runs_clean[['distance_km','moving_time_min','avg_pace_min_km','hr_mean']].dtypes)

                 date                   name  distance_km  moving_time_min  \
0 2022-04-29 23:49:01  First run in a while!         6.80        44.050000   
1 2022-05-15 12:17:25            Morning Run         5.87        37.200000   
2 2023-08-22 10:33:07            Morning Run        10.50        65.933333   
3 2023-08-23 09:00:38            Morning Run         6.72        54.716667   
4 2023-09-11 11:55:27        Garneau Parkrun         5.02        26.933333   
5 2023-09-20 12:45:16            Morning Run         5.22        29.733333   
6 2023-10-08 13:37:24            Morning Run         6.17        37.433333   
7 2023-10-21 14:03:23            Morning Run         5.08        29.633333   
8 2023-10-21 18:41:22              Lunch Run         5.12        35.516667   
9 2023-10-24 13:14:24            Morning Run         5.35        31.583333   

   avg_pace_min_km  hr_mean  
0         6.477523      NaN  
1         6.329915      NaN  
2         6.275100      NaN  
3         8.138021   

In [17]:
# ── Diagnose distance units ──────────────────────────────────────────────────
print("Raw distance sample (first 5):")
print(acts[acts['Activity Type'].isin(RUN_TYPES)]['Distance'].head())
print("\nThese look like km values (6.8, 5.87, 10.5...) not metres")
print("Strava exports Distance in km, not metres — skip the /1000 conversion")

Raw distance sample (first 5):
0     4.01
2     9.01
23    4.72
26    7.25
42    5.12
Name: Distance, dtype: str

These look like km values (6.8, 5.87, 10.5...) not metres
Strava exports Distance in km, not metres — skip the /1000 conversion


In [18]:
# ── Diagnose GPX speed ───────────────────────────────────────────────────────
sample_gpx = sorted((raw_dir / "activities").glob("*.gpx"))[0]
tree = ET.parse(sample_gpx)
root = tree.getroot()

NS = {
    'gpx':    'http://www.topografix.com/GPX/1/1',
    'gpxtpx': 'http://www.garmin.com/xmlschemas/TrackPointExtension/v1',
}

# Grab first 5 timestamps raw
points = root.findall('.//gpx:trkpt', NS)
for pt in points[:5]:
    t = pt.find('gpx:time', NS)
    print(repr(t.text))

'2023-10-21T14:03:23Z'
'2023-10-21T14:03:24Z'
'2023-10-21T14:03:25Z'
'2023-10-21T14:03:26Z'
'2023-10-21T14:03:27Z'


In [45]:
from tqdm import tqdm
import warnings

def get_activity_id_from_filename(filename_field):
    """
    Extract activity ID from the Filename column in activities.csv.
    Strava stores paths like 'activities/11147594123.fit.gz'
    We want just '11147594123'
    """
    if pd.isna(filename_field):
        return None
    p = Path(filename_field)
    # Strip all extensions: .fit.gz → stem twice, .gpx → stem once
    stem = p.stem
    if stem.endswith('.fit'):
        stem = stem[:-4]  # strip the .fit from .fit.gz
    return stem


def parse_any_activity(filepath):
    """
    Dispatch to the right parser based on file extension.
    Returns (activity_id, DataFrame) or (activity_id, None).
    """
    filepath = Path(filepath)
    name = filepath.name.lower()

    if ':zone.identifier' in name:
        return None, None  # Windows metadata junk

    if name.endswith('.fit.gz') or name.endswith('.fit'):
        return filepath.stem.replace('.fit', ''), parse_fit_file(filepath)
    elif name.endswith('.gpx'):
        return filepath.stem, parse_gpx_file(filepath)
    else:
        return None, None  # unknown format, skip


def parse_all_activities(activities_dir, runs_df):
    # Build set of file stems we want
    target_stems = set(runs_df['file_stem'].dropna().astype(str))
    print(f"Target runs: {len(target_stems)}")

    # Build map of stem → filepath for everything on disk
    all_files = [f for f in Path(activities_dir).iterdir()
             if ':Zone.Identifier' not in f.name]
    
    id_to_file = {}
    for f in all_files:  # all_files already excludes Zone.Identifier
        stem = f.name
        for ext in ['.fit.gz', '.fit', '.gpx']:
            if stem.endswith(ext):
                stem = stem[:-len(ext)]
                break
        else:
            continue  # skip files with no recognised extension
        id_to_file[stem] = f  # safe — Zone.Identifier already filtered out

    parseable = [(stem, id_to_file[stem]) for stem in target_stems if stem in id_to_file]
    missing   = [stem for stem in target_stems if stem not in id_to_file]
    print(f"Parseable: {len(parseable)}  |  Missing: {len(missing)}")

    results = []
    failed  = []

    for file_stem, filepath in tqdm(parseable, desc="Parsing runs"):
        try:
            _, df = parse_any_activity(filepath)
            if df is not None:
                # Use file_stem as the activity identifier in the stream data
                # so it joins cleanly back to runs_clean later
                df['activity_id'] = file_stem
                results.append(df)
            else:
                failed.append((file_stem, "returned None"))
        except Exception as e:
            failed.append((file_stem, str(e)))

    print(f"\nSuccessfully parsed: {len(results)}")
    print(f"Failed/skipped:      {len(failed)}")
    if failed:
        print(f"  Failures: {failed[:5]}")

    return results, failed


streams_list, failures = parse_all_activities(
    raw_dir / "activities",
    runs_clean
)

Target runs: 211
Parseable: 211  |  Missing: 0


Parsing runs:   9%|█████▊                                                              | 18/211 [00:06<01:26,  2.24it/s]

  dist_m[:5]: [0.    3.678 1.916 3.448 3.7  ]
  dt[:5]:     [1 1 1 1 1]
  raw m/s[:5]: [0.    3.678 1.916 3.448 3.7  ]


Parsing runs:   9%|██████▍                                                             | 20/211 [00:08<02:01,  1.57it/s]

  dist_m[:5]: [0.    1.459 1.112 2.446 1.009]
  dt[:5]:     [1 1 1 1 1]
  raw m/s[:5]: [0.    1.459 1.112 2.446 1.009]


...

  dist_m[:5]: [0.    1.606 1.336 1.12  2.205]
  dt[:5]:     [1 1 1 1 1]
  raw m/s[:5]: [0.    1.606 1.336 1.12  2.205]


Parsing runs:  96%|████████████████████████████████████████████████████████████████▍  | 203/211 [01:23<00:03,  2.60it/s]

  dist_m[:5]: [0.    0.812 0.554 0.554 0.683]
  dt[:5]:     [1 1 1 1 1]
  raw m/s[:5]: [0.    0.812 0.554 0.554 0.683]


Parsing runs: 100%|███████████████████████████████████████████████████████████████████| 211/211 [01:27<00:00,  2.42it/s]


Successfully parsed: 207
Failed/skipped:      4
  Failures: [('9886146103', 'returned None'), ('10000952539', 'returned None'), ('10119751760', 'returned None'), ('10119756807', 'returned None')]


In [46]:
streams_df = pd.concat(streams_list, axis=0)

print(f"Total stream rows:  {len(streams_df):,}")
print(f"Unique activities:  {streams_df['activity_id'].nunique()}")
print(f"Memory usage:       {streams_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nNull rates (%):")
print((streams_df.isna().sum() / len(streams_df) * 100).round(1))
# Save as CSV instead — parquet has a pyarrow conflict in this environment
# CSV is fine for this size (few MB), just slightly slower to reload
streams_df.to_csv("../data/processed/streams.csv")
runs_clean.to_csv("../data/processed/runs.csv", index=False)

print(f"Total stream rows:  {len(streams_df):,}")
print(f"Unique activities:  {streams_df['activity_id'].nunique()}")
print(f"Memory usage:       {streams_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nNull rates (%):")
print((streams_df.isna().sum() / len(streams_df) * 100).round(1))
print(f"\nStream IDs with no CSV match: {len(set(streams_df['activity_id'].unique()) - set(runs_clean['file_stem'].astype(str)))}")
print("\nSaved.")

Total stream rows:  142,103
Unique activities:  207
Memory usage:       35.8 MB

Null rates (%):
cadence              19.4
distance             19.4
enhanced_altitude     6.8
enhanced_speed        0.0
heart_rate           14.4
lat                   7.3
lon                   7.3
pace_min_km           1.6
activity_id           0.0
dtype: float64
Total stream rows:  142,103
Unique activities:  207
Memory usage:       35.8 MB

Null rates (%):
cadence              19.4
distance             19.4
enhanced_altitude     6.8
enhanced_speed        0.0
heart_rate           14.4
lat                   7.3
lon                   7.3
pace_min_km           1.6
activity_id           0.0
dtype: float64

Stream IDs with no CSV match: 0

Saved.


In [36]:
# Step 1: Look at what the Filename column actually contains
print("Sample Filename values from runs_clean:")
print(runs_clean['filename'].head(20).to_string())

Sample Filename values from runs_clean:
0         activities/7062035164.gpx
1         activities/7146896280.gpx
2         activities/9697773893.gpx
3         activities/9704255381.gpx
4         activities/9827502306.gpx
5         activities/9886146103.fit
6        activities/10000952539.fit
7        activities/10078375825.gpx
8        activities/10119756807.fit
9        activities/10119751760.fit
10       activities/10107949856.gpx
11       activities/10113345452.gpx
12       activities/10126159096.gpx
13       activities/10130645756.gpx
14       activities/10147948262.gpx
15       activities/10246786020.gpx
16       activities/10379368938.gpx
17       activities/10399960876.gpx
18    activities/11147594123.fit.gz
19    activities/11147642575.fit.gz


In [37]:
# Step 2: Look at what files actually exist on disk
all_files = list((raw_dir / "activities").iterdir())
non_meta = [f for f in all_files if ':Zone.Identifier' not in f.name]

print(f"\nTotal non-metadata files on disk: {len(non_meta)}")
print("\nFirst 20 filenames on disk:")
for f in sorted(non_meta)[:20]:
    print(f" {f.name}")


Total non-metadata files on disk: 701

First 20 filenames on disk:
 10000952539.fit
 10046118394.fit
 10078375825.gpx
 10107949856.gpx
 10113345452.gpx
 10119751760.fit
 10119756807.fit
 10119757923.fit
 10119758809.fit
 10119759605.fit
 10119760536.fit
 10126159096.gpx
 10130645756.gpx
 10147948262.gpx
 10166947256.fit
 10166948421.fit
 10246786020.gpx
 10379368938.gpx
 10399960876.gpx
 11147594123.fit.gz


In [38]:
# Step 3: Compare — what does the CSV say vs what's on disk?
# Build the disk set
disk_stems = set()
for f in non_meta:
    stem = f.stem
    if stem.endswith('.fit'):
        stem = stem[:-4]
    disk_stems.add(stem)

# Build the CSV set
csv_ids = set(runs_clean['activity_id'].astype(str))

print(f"\nRun IDs in CSV:    {len(csv_ids)}")
print(f"Files on disk:     {len(disk_stems)}")
print(f"In both:           {len(csv_ids & disk_stems)}")
print(f"In CSV only:       {len(csv_ids - disk_stems)}")
print(f"On disk only:      {len(disk_stems - csv_ids)}")

# Show a few from each mismatch category
print(f"\nExample CSV-only IDs (in CSV but no file): {list(csv_ids - disk_stems)[:5]}")
print(f"Example disk-only (file but no CSV entry): {list(disk_stems - csv_ids)[:5]}")


Run IDs in CSV:    211
Files on disk:     701
In both:           18
In CSV only:       193
On disk only:      683

Example CSV-only IDs (in CSV but no file): ['11302938511', '10743445821', '10630393180', '10933429465', '11850553595']
Example disk-only (file but no CSV entry): ['18066704144', '15977135111', '11740994027', '15040919250', '11344097974']


In [39]:
def extract_file_stem(filename_str):
    """
    Extract the numeric stem from a Filename field like 
    'activities/11147594123.fit.gz' → '11147594123'
    """
    if pd.isna(filename_str):
        return None
    p = Path(filename_str).name   # '11147594123.fit.gz'
    # Strip extensions one at a time until we hit the number
    stem = p
    for ext in ['.fit.gz', '.fit', '.gpx']:
        if stem.endswith(ext):
            stem = stem[:-len(ext)]
            break
    return stem

# Add a file_stem column — this matches what's actually on disk
runs_clean['file_stem'] = runs_clean['filename'].apply(extract_file_stem)

# Verify the fix
csv_stems = set(runs_clean['file_stem'].dropna().astype(str))
print(f"Run file stems from CSV: {len(csv_stems)}")
print(f"Files on disk:           {len(disk_stems)}")
print(f"In both:                 {len(csv_stems & disk_stems)}")
print(f"Still missing:           {len(csv_stems - disk_stems)}")
print(f"\nSample file_stem values:")
print(runs_clean[['activity_id','filename','file_stem']].head(10).to_string())

Run file stems from CSV: 211
Files on disk:           701
In both:                 211
Still missing:           0

Sample file_stem values:
   activity_id                    filename    file_stem
0   7062035164   activities/7062035164.gpx   7062035164
1   7146896280   activities/7146896280.gpx   7146896280
2   9697773893   activities/9697773893.gpx   9697773893
3   9704255381   activities/9704255381.gpx   9704255381
4   9827502306   activities/9827502306.gpx   9827502306
5   9886146103   activities/9886146103.fit   9886146103
6  10000952539  activities/10000952539.fit  10000952539
7  10078375825  activities/10078375825.gpx  10078375825
8  10119756807  activities/10119756807.fit  10119756807
9  10119751760  activities/10119751760.fit  10119751760


In [49]:
# Inspect the 4 failed files
failed_stems = ['9886146103', '10000952539', '10119751760', '10119756807']

for stem in failed_stems:
    # Find the file
    matches = list((raw_dir / "activities").glob(f"{stem}*"))
    if not matches:
        print(f"{stem}: no file found")
        continue
    
    filepath = matches[0]
    print(f"\n{'─'*50}")
    print(f"File: {filepath.name}  ({filepath.stat().st_size} bytes)")
    
    # Check what's in the CSV for this activity
    row = runs_clean[runs_clean['file_stem'] == stem]
    if not row.empty:
        print(f"CSV: {row['name'].values[0]}  |  {row['distance_km'].values[0]:.2f} km  |  {row['moving_time_min'].values[0]:.1f} min")
    
    # Try to open and count records
    fit = fitparse.FitFile(str(filepath))
    msg_counts = {}
    for msg in fit.get_messages():
        msg_counts[msg.name] = msg_counts.get(msg.name, 0) + 1
    print(f"Message counts: { {k:v for k,v in msg_counts.items() if v > 0} }")
    


──────────────────────────────────────────────────
File: 9886146103.fit:Zone.Identifier  (25 bytes)
CSV: Morning Run  |  5.22 km  |  29.7 min


FitHeaderError: Invalid .FIT File Header

# Summary

In [48]:
# ── Data quality summary — add this as the last cell ─────────────────────────

print("=" * 55)
print("PHASE 1 COMPLETE — DATA QUALITY NOTES")
print("=" * 55)

print(f"""
RUNS SUMMARY (runs.csv)
  Total runs:          {len(runs_clean)}
  Date range:          {runs_clean['date'].min().date()} → {runs_clean['date'].max().date()}
  Distance range:      {runs_clean['distance_km'].min():.1f} – {runs_clean['distance_km'].max():.1f} km
  Null HR (summary):   {runs_clean['hr_mean'].isna().sum()} / {len(runs_clean)} runs

STREAMS (streams.csv)  
  Total rows:          {len(streams_df):,}
  Parsed activities:   {streams_df['activity_id'].nunique()} / {len(runs_clean)}
  Failed (Zone.ID bug): 4  → re-run after Zone.Identifier fix

NULL RATES IN STREAMS
  heart_rate:    14.4%  → early runs, no HR strap. Flag, do not impute.
  cadence:       19.4%  → GPX phone recordings. Expected.
  lat/lon:        7.3%  → treadmill/indoor runs.
  altitude:       6.8%  → indoor or no barometer.

KNOWN ISSUES
  - pace_min_km derived from GPS for GPX files (noisy at walk/stop)
  - .fit files use enhanced_speed directly (more reliable)
  - 4 activities failed due to :Zone.Identifier clobber bug
  - activity_id in streams = file_stem (NOT Activity ID from CSV)
    → always join on file_stem column in runs.csv
""")

PHASE 1 COMPLETE — DATA QUALITY NOTES

RUNS SUMMARY (runs.csv)
  Total runs:          211
  Date range:          2022-04-29 → 2026-04-12
  Distance range:      0.4 – 25.1 km
  Null HR (summary):   13 / 211 runs

STREAMS (streams.csv)  
  Total rows:          142,103
  Parsed activities:   207 / 211
  Failed (Zone.ID bug): 4  → re-run after Zone.Identifier fix

NULL RATES IN STREAMS
  heart_rate:    14.4%  → early runs, no HR strap. Flag, do not impute.
  cadence:       19.4%  → GPX phone recordings. Expected.
  lat/lon:        7.3%  → treadmill/indoor runs.
  altitude:       6.8%  → indoor or no barometer.

KNOWN ISSUES
  - pace_min_km derived from GPS for GPX files (noisy at walk/stop)
  - .fit files use enhanced_speed directly (more reliable)
  - 4 activities failed due to :Zone.Identifier clobber bug
  - activity_id in streams = file_stem (NOT Activity ID from CSV)
    → always join on file_stem column in runs.csv

